# Reporte Stage 2 — Wave roi3ch (IL-CLDM)

Comparación de **seis** experimentos roi3ch en split primario: baseline **`h05_roi3ch`**, baseline calibrado **`calib_roi3ch_uni`**, variante calibrada **`h05_calib_roi3ch`**, dos variantes warm-start desde `h05_roi3ch`, y el perfil ROI patch **`calib_roi3ch_uni_rois`** (`data_rois/`). Fuente: `experiments/registry.json`.

| Clave | Rol | Delta único (vs baseline) |
|-------|-----|---------------------------|
| `h05_roi3ch` | baseline | ROI 3.er canal UNet + high-t 0.5 |
| `calib_roi3ch_uni` | baseline calibrado | calibración latente `(z−μ)/σ`, **t uniforme**, ROI 3ch, scratch |
| `h05_calib_roi3ch` | variante calibrada | warm-start `calib_roi3ch_uni` + high-t 0.5 + ROI 3ch |
| `roi3ch_nolabel` | variante | `USE_TRAIN_LABEL=0` |
| `roi3ch_ht08` | variante | `HIGH_T_SAMPLE_PROB=0.8` |
| `calib_roi3ch_uni_rois` | ROI patch | mismo recipe calibrada en `data_rois/` (105³ → working 108×104×104) |

**Alcance:** split **test** primario (n=36). Export CSV: `result/stage2_roi3ch_wave_summary/`.

**Métrica primaria:** balanced accuracy en clasificación de latentes. PSNR/SSIM secundarios.

Secciones:
1. Convergencia en entrenamiento
2. Reconstrucción en test (inferencia) + métricas ROI
3. Visualizaciones comparables (mismos sujetos)
4. Clasificación en test
5. Síntesis y exportación
5b. Visualización de latentes test (mejor clasificador: PCA / t-SNE / UMAP)
5c. Probabilidad Parkinson vs estadio NHY (mejor clasificador)
6. Diagnóstico sampling (`t` fijo vs DDIM 100)

**Apéndice A.** Proceso de difusión (baseline `h05_roi3ch`): schedule forward + denoising UNet


## 0. Configuración

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID";
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, here.parent):
        if (candidate / "result").is_dir() and (candidate / "code" / "stage2_report_utils.py").is_file():
            return candidate
    raise RuntimeError(
        "No se encontró la raíz del repo. Ejecuta el notebook desde IL-CLDM/ o IL-CLDM/code/."
    )


ROOT = find_repo_root()
os.chdir(ROOT)

root_str = str(ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

import importlib

for _mod_name in list(sys.modules):
    if _mod_name in ("code", "code.stage2_report_utils") or _mod_name.endswith("stage2_report_utils"):
        del sys.modules[_mod_name]
if "code" in sys.modules and not hasattr(sys.modules.get("code"), "__path__"):
    del sys.modules["code"]

import code.stage2_report_utils as stage2_report_utils
importlib.reload(stage2_report_utils)
from code.stage2_report_utils import (
    LABEL_NAMES,
    available_comparison,
    load_classification_metrics_for,
    load_inference_reconstruction_for,
    load_registry,
    load_synthetic_volume_for,
    load_test_ids,
    load_test_latent_matrix_for,
    build_parkinson_probability_by_nhy_stage,
    NHY_STAGE_ORDER,
    load_training_frames_for,
    manifest_summary_table,
    pick_stratified_subjects,
    repo_root,
    training_checkpoint_summary,
    validate_comparison,
    _sanitize_training_csv,
    domain_buckets_for_modes,
    subset_by_modes,
    short_labels_for_modes,
    colors_for_modes,
    modes_for_profile,
)
from viz_pairs import display_range, load_cropped

# --- Wave roi3ch: constantes y helpers locales (solo notebook) ---
WAVE_BASELINE = "h05_roi3ch"
WAVE_ORDER = (
    WAVE_BASELINE,
    "calib_roi3ch_uni",
    #"h05_calib_roi3ch",
    #"roi3ch_nolabel",
    #"roi3ch_ht08",
    "calib_roi3ch_uni_rois",
)
STAGE2_WAVE_EXPORT_DIR = "stage2_roi3ch_wave_summary"
_REGISTRY = load_registry()["experiments"]


def _mode_split_profile(mode_key: str) -> str:
    return _REGISTRY[mode_key].get("split_profile", "primary")


def data_root_for_mode(mode_key: str) -> Path:
    if _mode_split_profile(mode_key) == "data_rois":
        return repo_root() / "data_rois"
    return repo_root() / "data"


def load_gt_spect_for_mode(mode_key: str, subject_id: str) -> np.ndarray:
    root = data_root_for_mode(mode_key) / "whole_Abeta"
    if _mode_split_profile(mode_key) == "data_rois":
        from dataset import nifti_to_numpy, resolve_nifti_path

        return nifti_to_numpy(str(resolve_nifti_path(str(root), subject_id)), zscore=True)
    return load_cropped(root, subject_id)


def _subset_wave(pool, order):
    return {k: pool[k] for k in order if k in pool}


def _with_registry_meta(comparison):
    out = {}
    for key, cfg in comparison.items():
        entry = dict(cfg)
        reg = _REGISTRY[key]
        entry["short_label"] = reg.get("short_label", key)
        entry["label"] = reg.get("label", entry.get("label", key))
        entry["early_stop"] = reg.get("early_stop", entry.get("early_stop", "image"))
        entry["split_profile"] = reg.get("split_profile", "primary")
        out[key] = entry
    return out


def available_wave_comparison(pool=None):
    return _with_registry_meta(_subset_wave(pool or available_comparison(), WAVE_ORDER))


def load_test_ids_for_mode(mode_key: str) -> list[str]:
    split = _REGISTRY[mode_key].get("split_profile", "primary")
    if split == "nhy_v1":
        path = repo_root() / "data_info" / "splits" / "nhy_v1" / "test.txt"
    else:
        path = repo_root() / "data_info" / "test.txt"
    return [line.strip() for line in path.read_text().splitlines() if line.strip()]


def validate_wave_comparison(pool=None):
    warnings: list[str] = []
    comparison = available_wave_comparison(pool)
    for mode_key, cfg in comparison.items():
        exp_name = cfg["exp"]
        run_id = cfg["run_id"]
        test_ids = set(load_test_ids_for_mode(mode_key))
        root = repo_root() / "result" / exp_name
        for fname in ("loss_curve.csv", "validation.csv", "test.csv"):
            if not (root / fname).exists():
                warnings.append(f"Missing {root / fname}")
        inf = repo_root() / "result" / exp_name / "inference" / run_id
        recon_path = inf / "reconstruction_metrics.csv"
        if not recon_path.exists():
            warnings.append(f"Missing {recon_path}")
            continue
        recon = pd.read_csv(recon_path, dtype={"ID": str})
        test_recon = recon[recon["split"] == "test"]
        if len(test_recon) != len(test_ids):
            warnings.append(
                f"{exp_name} ({run_id}): expected {len(test_ids)} test rows, got {len(test_recon)}"
            )
        cls_path = inf / "classification" / "classification_metrics.csv"
        if not cls_path.exists():
            warnings.append(f"Missing {cls_path}")
    return warnings


def load_training_frames_wave(exp_name: str):
    frames = load_training_frames_for(exp_name)
    cls_path = repo_root() / "result" / exp_name / "validation_cls.csv"
    if cls_path.is_file():
        frames["validation_cls"] = pd.read_csv(cls_path)
    return frames


def checkpoint_summary_for_mode(mode, cfg, frames):
    criterion = cfg.get("early_stop", "image")
    if criterion == "cls" and frames.get("validation_cls") is not None and not frames["validation_cls"].empty:
        val_cls = frames["validation_cls"]
        best_epoch = int(val_cls.loc[val_cls["balanced_accuracy"].idxmax(), "Epoch"])
        summary = training_checkpoint_summary(
            frames["validation"], frames["test"], criterion="image"
        )
        summary["best_epoch"] = best_epoch
        summary["early_stop_criterion"] = "cls"
        summary["val_balanced_accuracy"] = float(
            val_cls.loc[val_cls["balanced_accuracy"].idxmax(), "balanced_accuracy"]
        )
        val_row = frames["validation"][frames["validation"]["Epoch"] == best_epoch]
        if len(val_row):
            summary["val_psnr"] = float(val_row["PSNR"].iloc[0])
            summary["val_ssim"] = float(val_row["SSIM"].iloc[0])
        test_row = frames["test"][frames["test"]["Epoch"] == best_epoch]
        if len(test_row):
            summary["test_psnr_at_best"] = float(test_row["PSNR"].iloc[0])
            summary["test_ssim_at_best"] = float(test_row["SSIM"].iloc[0])
    else:
        summary = training_checkpoint_summary(
            frames["validation"],
            frames["test"],
            criterion=criterion if criterion in {"image", "latent"} else "image",
            validation_latent=frames.get("validation_latent"),
            test_latent=frames.get("test_latent"),
        )
        summary["val_balanced_accuracy"] = float("nan")
    summary["mode"] = mode
    summary["mode_short"] = cfg["short_label"]
    summary["mode_label"] = cfg["label"]
    return summary


_all_available = available_comparison()
COMPARISON = available_wave_comparison(_all_available)
MODES = list(COMPARISON.keys())
if len(MODES) != len(WAVE_ORDER):
    missing = [m for m in WAVE_ORDER if m not in COMPARISON]
    raise RuntimeError(f"Faltan resultados de entrenamiento para: {missing}")

MODE_LABELS = {k: COMPARISON[k]["label"] for k in MODES}
MODE_SHORT = {k: COMPARISON[k]["short_label"] for k in MODES}
MODE_ORDER = [MODE_LABELS[m] for m in MODES]
MODE_SHORT_ORDER = [MODE_SHORT[m] for m in MODES]
_PALETTE = ["#4C72B0", "#8172B2", "#DD8452", "#55A868", "#C44E52", "#9467BD"]
MODE_COLORS = {m: _PALETTE[i % len(_PALETTE)] for i, m in enumerate(MODES)}
MODE_PALETTE = [MODE_COLORS[m] for m in MODES]
REPORT_EXPORT_DIR = repo_root() / "result" / STAGE2_WAVE_EXPORT_DIR
DIAG_MODES_DEFAULT = list(MODES)
PRIMARY_SPLIT_MODES = [m for m in MODES if _mode_split_profile(m) not in {"nhy_v1", "data_rois"}]
FULLBRAIN_PRIMARY_MODES = [m for m in MODES if _mode_split_profile(m) == "primary"]
ROI_PATCH_MODES = [m for m in MODES if _mode_split_profile(m) == "data_rois"]
NHY_SPLIT_MODES = [m for m in MODES if _mode_split_profile(m) == "nhy_v1"]

DATA_ROOT = repo_root() / "data"
TEST_IDS = load_test_ids()

sns.set_theme(style="whitegrid", context="notebook")
%matplotlib inline

print(f"Repo root: {repo_root()}")
print(f"Test subjects (split primario): {len(TEST_IDS)}")
if NHY_SPLIT_MODES:
    for m in NHY_SPLIT_MODES:
        print(f"Test subjects ({MODE_SHORT[m]}, split NHY): {len(load_test_ids_for_mode(m))}")
print(f"Experimentos en el reporte: {len(MODES)} (baseline: {WAVE_BASELINE})")
if ROI_PATCH_MODES:
    print(f"Perfil ROI patch: {ROI_PATCH_MODES}")
for mode in MODES:
    cfg = COMPARISON[mode]
    print(f"  [{MODE_SHORT[mode]}] {MODE_LABELS[mode]}: {cfg['exp']} / {cfg['run_id']}")
print(f"Export: {REPORT_EXPORT_DIR}")
for w in validate_wave_comparison(_all_available):
    print(f"WARNING: {w}")


## 0b. Resumen de configuración (registro + manifests)

Fuente única de verdad: `experiments/registry.json`. Cada corrida escribe `result/<exp>/run_manifest.json` (flags activos, seed, commit, GPU).

**Criterios de éxito / fallo** (relativos a `h05_roi3ch`, test n=36). `calib_roi3ch_uni` es baseline alternativo (calibración + t uniforme) — comparar métricas absolutas y vs `h05_roi3ch`. `h05_calib_roi3ch` se evalúa también vs `calib_roi3ch_uni`.

`calib_roi3ch_uni_rois` usa el mismo recipe calibrado en parches ROI (`data_rois/`, `split_profile=data_rois`). Sus PSNR/SSIM **no son directamente comparables** con los experimentos full-brain de esta wave; se reportan en tablas y visualización aparte.

| Variante | Cambio único | ÉXITO | FALLO |
|----------|--------------|-------|-------|
| `calib_roi3ch_uni` | latent calib + t uniforme + ROI 3ch | referencia E2 calibrada | — |
| `h05_calib_roi3ch` | warm-start calib + high-t 0.5 | bal.acc ≥ `calib_roi3ch_uni` **y** PSNR ≥ calib −0.2 dB | sin mejora vs calib **o** PSNR cae >0.3 dB |
| `roi3ch_nolabel` | `USE_TRAIN_LABEL=0` | bal.acc ≥ baseline **y** PSNR ≥ baseline −0.2 dB | bal.acc cae **o** PSNR cae >0.5 dB |
| `roi3ch_ht08` | `HIGH_T_SAMPLE_PROB=0.8` | bal.acc ≥ baseline **y** PSNR ≥ baseline −0.2 dB | sin mejora en clasificación **o** PSNR cae >0.3 dB |
| `calib_roi3ch_uni_rois` | recipe calib en `data_rois/` | referencia ROI patch | — |


In [ ]:
config_table = manifest_summary_table()
config_table = config_table[config_table["mode"].isin(MODES)].copy()
print(f"Experimentos en el reporte: {len(config_table)}")
print(f"Con manifest: {int(config_table['has_manifest'].sum())}")
_config_cols = [
    "mode", "short_label", "parent", "overrides", "high_t_prob", "use_train_label",
    "unet_roi_channel", "aux_every_n", "aux_lambda", "init_checkpoint", "has_manifest",
]
config_table[[c for c in _config_cols if c in config_table.columns]]


## 1. Convergencia durante entrenamiento

Fuentes en `result/<exp>/`:
- `loss_curve.csv`: MSE difusión (+ `aux_image_loss` si aplica; `latent_loss` solo si `LATENT_LOSS=1`)
- `validation_latent.csv`: MSE/MAE latente en validación (~cada 5 épocas) — panel latente cuando no hay `latent_loss` en train
- `validation.csv` / `test.csv`: PSNR/SSIM imagen (early stop por imagen en todos los experimentos de esta wave)

Curvas agrupadas **por dominio** (full-brain vs ROI patch vs NHY): no mezclar experimentos de distinto `split_profile` en el mismo panel.


In [ ]:
from matplotlib.ticker import MaxNLocator

training = {mode: load_training_frames_wave(COMPARISON[mode]["exp"]) for mode in MODES}


def _series_values(series_list: list[pd.Series]) -> np.ndarray:
    parts = [s.dropna().astype(float).values for s in series_list if len(s)]
    if not parts:
        return np.array([])
    vals = np.concatenate(parts)
    return vals[np.isfinite(vals)]


def _use_log_scale(series_list: list[pd.Series], ratio_threshold: float = 15.0) -> bool:
    vals = _series_values(series_list)
    pos = vals[vals > 0]
    if len(pos) < 2:
        return False
    return pos.max() / pos.min() > ratio_threshold


def _style_loss_axis(ax, *, log_y: bool = False) -> None:
    if log_y:
        ax.set_yscale("log")
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6, prune="both"))
    ax.tick_params(axis="x", labelsize=9)
    ax.tick_params(axis="y", labelsize=9)


has_aux = any(
    "aux_image_loss" in training[m].get("loss_curve", pd.DataFrame()).columns
    for m in MODES
)
has_train_latent = any(
    "latent_loss" in training[m].get("loss_curve", pd.DataFrame()).columns
    for m in MODES
)
has_val_latent = any("validation_latent" in training[m] for m in MODES)
n_plots = 1 + int(has_aux) + int(has_train_latent or has_val_latent)

fig, axes = plt.subplots(1, n_plots, figsize=(5.5 * n_plots, 4.5))
if n_plots == 1:
    axes = [axes]

pi = 0
mse_series = [training[m]["loss_curve"]["MSE_loss"] for m in MODES]
for mode in MODES:
    loss = training[mode]["loss_curve"]
    axes[pi].plot(
        loss["Epoch"], loss["MSE_loss"],
        label=MODE_SHORT[mode], color=MODE_COLORS[mode], linewidth=1.5,
    )
log_mse = _use_log_scale(mse_series)
_style_loss_axis(axes[pi], log_y=log_mse)
axes[pi].set(
    xlabel="Época",
    ylabel="MSE loss (log)" if log_mse else "MSE loss",
    title="Pérdida de difusión (MSE)",
)
axes[pi].legend(fontsize=8, loc="best")
pi += 1

if has_aux:
    aux_series = []
    for mode in MODES:
        loss = training[mode]["loss_curve"]
        if "aux_image_loss" not in loss.columns:
            continue
        aux = loss["aux_image_loss"].dropna()
        if aux.empty:
            continue
        aux_series.append(aux)
        axes[pi].plot(
            loss["Epoch"], loss["aux_image_loss"],
            label=MODE_SHORT[mode], color=MODE_COLORS[mode], linewidth=1.5,
        )
    log_aux = _use_log_scale(aux_series)
    _style_loss_axis(axes[pi], log_y=log_aux)
    axes[pi].set(
        xlabel="Época",
        ylabel="Aux image loss (log)" if log_aux else "Aux image loss",
        title="Pérdida auxiliar de imagen",
    )
    axes[pi].legend(fontsize=8, loc="best")
    pi += 1

if has_train_latent or has_val_latent:
    latent_series = []
    if has_train_latent:
        for mode in MODES:
            loss = training[mode]["loss_curve"]
            if "latent_loss" not in loss.columns:
                continue
            latent_series.append(loss["latent_loss"])
            axes[pi].plot(
                loss["Epoch"], loss["latent_loss"],
                label=MODE_SHORT[mode], color=MODE_COLORS[mode], linewidth=1.5,
            )
        ylab = "Latent loss (train)"
        title = "Pérdida directa en latente (train)"
    else:
        for mode in MODES:
            if "validation_latent" not in training[mode]:
                continue
            vl = training[mode]["validation_latent"]
            latent_series.append(vl["latent_MSE"])
            axes[pi].plot(
                vl["Epoch"], vl["latent_MSE"],
                marker="o", ms=3, linewidth=1.2,
                label=MODE_SHORT[mode], color=MODE_COLORS[mode],
            )
        ylab = "latent_MSE (val)"
        title = "Métrica latente en validación (MSE)"
    log_lat = _use_log_scale(latent_series)
    _style_loss_axis(axes[pi], log_y=log_lat)
    axes[pi].set(
        xlabel="Época",
        ylabel=f"{ylab} (log)" if log_lat else ylab,
        title=title,
    )
    axes[pi].legend(fontsize=8, loc="best")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for mode in MODES:
    val = training[mode]["validation"]
    axes[0].plot(val["Epoch"], val["PSNR"], marker="o", ms=3, label=MODE_SHORT[mode], color=MODE_COLORS[mode])
    axes[1].plot(val["Epoch"], val["SSIM"], marker="o", ms=3, label=MODE_SHORT[mode], color=MODE_COLORS[mode])
for ax in axes:
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6, prune="both"))
axes[0].set(xlabel="Época", ylabel="PSNR (dB)", title="Validación — PSNR (imagen)")
axes[1].set(xlabel="Época", ylabel="SSIM", title="Validación — SSIM (imagen)")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

if any("validation_latent" in training[m] for m in MODES):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for mode in MODES:
        if "validation_latent" not in training[mode]:
            continue
        vl = training[mode]["validation_latent"]
        axes[0].plot(vl["Epoch"], vl["latent_MSE"], marker="o", ms=3, label=MODE_SHORT[mode], color=MODE_COLORS[mode])
        axes[1].plot(vl["Epoch"], vl["latent_MAE"], marker="o", ms=3, label=MODE_SHORT[mode], color=MODE_COLORS[mode])
    for ax in axes:
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6, prune="both"))
    axes[0].set(xlabel="Época", ylabel="latent_MSE", title="Validación — MSE latente")
    axes[1].set(xlabel="Época", ylabel="latent_MAE", title="Validación — MAE latente")
    for ax in axes:
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
ckpt_rows = []
for mode in MODES:
    cfg = COMPARISON[mode]
    summary = checkpoint_summary_for_mode(mode, cfg, training[mode])
    ckpt_rows.append(summary)

ckpt_df = pd.DataFrame(ckpt_rows)
cols = [
    "mode_short", "mode_label", "early_stop_criterion", "last_epoch", "best_epoch",
    "val_balanced_accuracy", "val_psnr", "val_ssim", "val_latent_mse", "val_latent_mae",
    "test_psnr_at_best", "test_ssim_at_best",
    "test_latent_mse_at_best", "test_latent_mae_at_best",
    "n_val_points", "n_test_checkpoints",
]
ckpt_df = ckpt_df[[c for c in cols if c in ckpt_df.columns]]
ckpt_df


## 2. Reconstrucción en test (inferencia)

Fuente: `result/<exp>/inference/<run_id>/reconstruction_metrics.csv` (`split == test`).
Incluye columnas opcionales `PSNR_roi` / `SSIM_roi` (ganglios basales).

**Dominios separados:** full-brain y ROI patch (`data_rois/`) se muestran en **filas distintas** de boxplots; no comparar métricas entre filas. Δ PSNR vs `h05_roi3ch` solo dentro del dominio full-brain.


In [ ]:
from matplotlib.ticker import MaxNLocator

_METRIC_YLABELS = {
    "PSNR": "PSNR (dB)",
    "SSIM": "SSIM",
    "PSNR_roi": "PSNR ROI (dB)",
    "SSIM_roi": "SSIM ROI",
}


def _boxplot_ylim(series: pd.Series) -> tuple[float, float]:
    y = pd.to_numeric(series, errors="coerce").dropna().astype(float)
    if y.empty:
        return 0.0, 1.0
    q1, q3 = np.percentile(y, [25, 75])
    iqr = q3 - q1
    if iqr <= 0:
        pad = max(abs(y.mean()) * 0.1, 0.5)
        return float(y.min() - pad), float(y.max() + pad)
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    pad = 0.06 * (hi - lo)
    return float(lo - pad), float(hi + pad)


def _style_recon_box_axis(ax, *, col: str, title: str) -> None:
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("")
    ax.set_ylabel(_METRIC_YLABELS.get(col, col))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune="both"))
    ax.tick_params(axis="x", rotation=40, labelsize=8)
    ax.tick_params(axis="y", labelsize=9)


def _metric_plot_specs(recon_df: pd.DataFrame) -> list[tuple[str, str]]:
    specs = [("PSNR", "PSNR en test"), ("SSIM", "SSIM en test")]
    if "PSNR_roi" in recon_df.columns:
        specs.extend([("PSNR_roi", "PSNR ROI"), ("SSIM_roi", "SSIM ROI")])
    return specs


def plot_recon_boxplot_grid(recon_df: pd.DataFrame, *, suptitle: str | None = None) -> None:
    buckets = domain_buckets_for_modes(MODES, _REGISTRY)
    if not buckets:
        print("No hay dominios activos para boxplots.")
        return

    plot_metrics = _metric_plot_specs(subset_by_modes(recon_df, buckets[0][1]))
    n_metrics = len(plot_metrics)
    n_rows = len(buckets)
    fig, axes = plt.subplots(n_rows, n_metrics, figsize=(6 * n_metrics, 4.8 * n_rows), squeeze=False)

    for row_idx, (_key, modes, domain_label) in enumerate(buckets):
        df = subset_by_modes(recon_df, modes)
        order = short_labels_for_modes(modes, MODE_SHORT)
        palette = colors_for_modes(modes, MODE_COLORS)
        for col_idx, (col, metric_title) in enumerate(plot_metrics):
            ax = axes[row_idx, col_idx]
            if col not in df.columns or df[col].dropna().empty:
                ax.set_visible(False)
                continue
            sns.boxplot(data=df, x="mode_short", y=col, order=order, ax=ax, palette=palette)
            ylo, yhi = _boxplot_ylim(df[col])
            ax.set_ylim(ylo, yhi)
            title = metric_title if row_idx == 0 else ""
            _style_recon_box_axis(ax, col=col, title=title)
            if col_idx == 0:
                ax.set_ylabel(f"{domain_label}\n{_METRIC_YLABELS.get(col, col)}")

    if suptitle:
        fig.suptitle(suptitle, y=1.02)
    plt.tight_layout()
    plt.show()


def plot_recon_by_label_grid(recon_df: pd.DataFrame, *, metric: str = "PSNR") -> None:
    buckets = domain_buckets_for_modes(MODES, _REGISTRY)
    if not buckets:
        return

    n_rows = len(buckets)
    fig, axes = plt.subplots(n_rows, 2, figsize=(13, 4.8 * n_rows), squeeze=False)
    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, (_key, modes, domain_label) in enumerate(buckets):
        df = subset_by_modes(recon_df, modes)
        order = short_labels_for_modes(modes, MODE_SHORT)
        palette = colors_for_modes(modes, MODE_COLORS)
        for col_idx, label_name in enumerate(["control", "parkinson"]):
            ax = axes[row_idx, col_idx]
            subset = df[df["label_name"] == label_name]
            if subset.empty:
                ax.set_visible(False)
                continue
            sns.boxplot(data=subset, x="mode_short", y=metric, order=order, ax=ax, palette=palette)
            ylo, yhi = _boxplot_ylim(subset[metric])
            ax.set_ylim(ylo, yhi)
            _style_recon_box_axis(ax, col=metric, title=f"{metric} — {label_name} ({domain_label})")
    plt.tight_layout()
    plt.show()


recon_by_mode = {
    mode: load_inference_reconstruction_for(
        COMPARISON[mode]["exp"], COMPARISON[mode]["run_id"], mode
    )
    for mode in MODES
}
recon_all = pd.concat(recon_by_mode.values(), ignore_index=True)
recon_all["mode_label"] = recon_all["mode"].map(MODE_LABELS)
recon_all["mode_short"] = recon_all["mode"].map(MODE_SHORT)

metric_cols = ["PSNR", "SSIM"]
if "PSNR_roi" in recon_all.columns:
    metric_cols.extend(["PSNR_roi", "SSIM_roi"])

for domain_key, modes, domain_label in domain_buckets_for_modes(MODES, _REGISTRY):
    print(f"Resumen reconstrucción — {domain_label}")
    display(
        subset_by_modes(recon_all, modes)
        .groupby("mode_label")[metric_cols]
        .agg(["mean", "std", "min", "max"])
        .round(3)
    )

summary_recon = (
    recon_all.groupby("mode_label")[metric_cols]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)
summary_recon


In [ ]:
plot_recon_boxplot_grid(
    recon_all,
    suptitle="Reconstrucción en test — dominios separados (no comparar filas entre sí)",
)


In [ ]:
if WAVE_BASELINE in FULLBRAIN_PRIMARY_MODES:
    baseline_psnr = recon_all.loc[recon_all["mode"] == WAVE_BASELINE, "PSNR"].mean()
    primary_recon = subset_by_modes(recon_all, FULLBRAIN_PRIMARY_MODES)
    delta_psnr = (
        primary_recon.groupby("mode_short")["PSNR"].mean() - baseline_psnr
    ).reindex(short_labels_for_modes(FULLBRAIN_PRIMARY_MODES, MODE_SHORT)).round(3)
    print(f"Δ PSNR vs {MODE_SHORT[WAVE_BASELINE]} (media test, full-brain):")
    print(delta_psnr.to_string())
else:
    print(f"Baseline {WAVE_BASELINE} no está en full-brain; omitiendo Δ PSNR cross-domain.")

for _key, modes, domain_label in domain_buckets_for_modes(MODES, _REGISTRY):
    means = (
        subset_by_modes(recon_all, modes)
        .groupby("mode_short")[metric_cols]
        .mean()
        .round(3)
    )
    print(f"Medias test — {domain_label}:")
    print(means.to_string())

if NHY_SPLIT_MODES:
    for m in NHY_SPLIT_MODES:
        mean_psnr = recon_all.loc[recon_all["mode"] == m, "PSNR"].mean()
        print(f"{MODE_SHORT[m]} (split NHY): PSNR media test = {mean_psnr:.3f} dB")

In [ ]:
plot_recon_by_label_grid(recon_all, metric="PSNR")


In [ ]:
for domain_key, modes, domain_label in domain_buckets_for_modes(MODES, _REGISTRY):
    pivot = subset_by_modes(recon_all, modes).pivot(index="ID", columns="mode", values="PSNR")
    pivot["mean"] = pivot.mean(axis=1)
    pivot = pivot.sort_values("mean", ascending=False)
    pivot.columns = [MODE_LABELS.get(c, c) for c in pivot.columns]
    print(f"Top PSNR por sujeto — {domain_label}")
    display(pivot.head(10))


In [ ]:
for domain_key, modes, domain_label in domain_buckets_for_modes(MODES, _REGISTRY):
    if WAVE_BASELINE not in modes:
        print(f"Omitiendo scatter vs baseline en {domain_label} (baseline no pertenece al dominio).")
        continue
    variants = [m for m in modes if m != WAVE_BASELINE]
    if not variants:
        continue

    baseline = recon_by_mode[WAVE_BASELINE].set_index("ID")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, col in zip(axes, ["PSNR", "SSIM"]):
        for mode in variants:
            other = recon_by_mode[mode].set_index("ID")
            common = baseline.index.intersection(other.index)
            x = baseline.loc[common, col]
            y = other.loc[common, col]
            ax.scatter(x, y, alpha=0.75, label=MODE_SHORT[mode], color=MODE_COLORS[mode])
        lims = [
            min(baseline[col].min(), min(recon_by_mode[m][col].min() for m in variants)) - 0.5,
            max(baseline[col].max(), max(recon_by_mode[m][col].max() for m in variants)) + 0.5,
        ]
        ax.plot(lims, lims, "k--", alpha=0.4)
        ax.set(
            xlabel=f"{col} {MODE_SHORT[WAVE_BASELINE]}",
            ylabel=f"{col} variante",
            title=f"{col} por sujeto — {domain_label}",
        )
        ax.legend(fontsize=8, loc="lower right")
    fig.suptitle(f"Variantes vs baseline ({domain_label})", y=1.02)
    plt.tight_layout()
    plt.show()


## 3. Visualizaciones comparables

**Full-brain (split primario):** 6 sujetos test — 3 control + 3 parkinson (p10/p50/p90 PSNR medio entre modos full-brain). Columnas: GT SPECT | cada modo (corte axial central).

**ROI patch (`data_rois/`):** misma selección estratificada pero solo para `calib_roi3ch_uni_rois` (volúmenes 105×102×102, no mezclar con full-brain).


In [ ]:
recon_by_mode_fullbrain = {m: recon_by_mode[m] for m in FULLBRAIN_PRIMARY_MODES}
selected = pick_stratified_subjects(recon_by_mode_fullbrain)
selected[["ID", "label_name", "tier", "psnr_mean"] + [c for c in selected.columns if c.startswith("psnr_")]]


In [ ]:
def plot_comparison_grid(selected_df: pd.DataFrame, modes: list[str], *, title: str) -> None:
    col_titles = ["GT SPECT"] + [MODE_LABELS[m] for m in modes]
    n_cols = len(col_titles)
    n_rows = len(selected_df)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 3.2 * n_rows))
    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, (_, row) in enumerate(selected_df.iterrows()):
        sid = str(row["ID"])
        gt = load_gt_spect_for_mode(modes[0], sid)
        z = gt.shape[2] // 2
        gt_lim = display_range(gt, symmetric=False, percentiles=(5.0, 99.5))

        volumes = [gt]
        for mode in modes:
            cfg = COMPARISON[mode]
            volumes.append(load_synthetic_volume_for(cfg["exp"], cfg["run_id"], sid))

        for col_idx, (vol, col_title) in enumerate(zip(volumes, col_titles)):
            ax = axes[row_idx, col_idx]
            lim = gt_lim if col_idx == 0 else display_range(vol, symmetric=False, percentiles=(5.0, 99.5))
            ax.imshow(vol[:, :, z].T, cmap="hot", origin="lower", vmin=lim[0], vmax=lim[1])
            if row_idx == 0:
                ax.set_title(col_title, fontsize=10)
            ax.axis("off")
            if col_idx == 0:
                ax.set_ylabel(f"{sid}\n{row['label_name']}\n{row['tier']}", fontsize=9)

        metrics_txt = " | ".join(
            f"{MODE_LABELS[m][:6]} PSNR={recon_by_mode[m].loc[recon_by_mode[m]['ID']==sid, 'PSNR'].iloc[0]:.1f}"
            for m in modes
        )
        fig.text(0.02, 1 - (row_idx + 0.92) / n_rows, metrics_txt, fontsize=8, va="top")

    fig.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.show()


plot_comparison_grid(
    selected,
    FULLBRAIN_PRIMARY_MODES,
    title="Reconstrucciones SPECT — wave roi3ch full-brain (test)",
)

if ROI_PATCH_MODES:
    selected_roi = pick_stratified_subjects({m: recon_by_mode[m] for m in ROI_PATCH_MODES})
    plot_comparison_grid(
        selected_roi,
        ROI_PATCH_MODES,
        title="Reconstrucciones SPECT — ROI patch data_rois (test)",
    )


## 4. Clasificación en test

Fuente única: `classification/classification_metrics.csv` por experimento.

Tablas y barplots **por dominio** (full-brain / ROI patch / NHY): latentes y splits pueden diferir entre perfiles.
Los clasificadores entrenan con latentes **train** y evalúan en latentes **test** generados por inferencia.

In [ ]:
cls_frames = [
    load_classification_metrics_for(COMPARISON[m]["exp"], COMPARISON[m]["run_id"], m)
    for m in MODES
]
cls_all = pd.concat(cls_frames, ignore_index=True)
cls_all["mode_label"] = cls_all["mode"].map(MODE_LABELS)
cls_all["mode_short"] = cls_all["mode"].map(MODE_SHORT)

_cls_metric_cols = [
    "accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc",
]
_cls_metric_cols = [c for c in _cls_metric_cols if c in cls_all.columns]

cls_pivot_tables = {}
for domain_key, modes, domain_label in domain_buckets_for_modes(MODES, _REGISTRY):
    sub = subset_by_modes(cls_all, modes)
    print(f"Clasificación test — {domain_label}")
    table = sub.pivot_table(
        index="model",
        columns="mode_label",
        values=_cls_metric_cols,
    ).round(3)
    cls_pivot_tables[domain_key] = table
    display(table)

cls_pivot = cls_all.pivot_table(
    index="model",
    columns="mode_label",
    values=_cls_metric_cols,
).round(3)
cls_pivot


In [ ]:
buckets = domain_buckets_for_modes(MODES, _REGISTRY)
n_rows = max(len(buckets), 1)
fig, axes = plt.subplots(n_rows, 1, figsize=(10, 5 * n_rows), squeeze=False)

for ax, (_key, modes, domain_label) in zip(axes[:, 0], buckets):
    sub = subset_by_modes(cls_all, modes)
    if sub.empty:
        ax.set_visible(False)
        continue
    sns.barplot(
        data=sub,
        x="model",
        y="roc_auc",
        hue="mode_short",
        hue_order=short_labels_for_modes(modes, MODE_SHORT),
        palette=colors_for_modes(modes, MODE_COLORS),
        ax=ax,
    )
    ax.set_title(f"ROC AUC en test — {domain_label}")
    ax.set_xlabel("Modelo")
    ax.set_ylabel("ROC AUC")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(title="Experimento", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
best_per_mode = (
    cls_all.sort_values(["roc_auc", "f1"], ascending=False)
    .groupby("mode_label", as_index=False)
    .first()
)
_best_cols = [
    "mode_label", "model", "accuracy", "balanced_accuracy",
    "precision", "recall", "f1", "roc_auc",
]
best_per_mode[[c for c in _best_cols if c in best_per_mode.columns]].round(3)


## 5. Síntesis y exportación

In [ ]:
out_dir = REPORT_EXPORT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

recon_summary = (
    recon_all.groupby(["mode", "mode_short", "mode_label", "label_name"])[metric_cols]
    .agg(["mean", "std", "count"])
    .round(4)
)
recon_summary.to_csv(out_dir / "reconstruction_test_summary.csv")

for domain_key, modes, _domain_label in domain_buckets_for_modes(MODES, _REGISTRY):
    subset_by_modes(recon_all, modes).groupby(
        ["mode", "mode_short", "mode_label", "label_name"]
    )[metric_cols].agg(["mean", "std", "count"]).round(4).to_csv(
        out_dir / f"reconstruction_test_summary_{domain_key}.csv"
    )

cls_all.to_csv(out_dir / "classification_comparison.csv", index=False)
for domain_key, modes, _domain_label in domain_buckets_for_modes(MODES, _REGISTRY):
    subset_by_modes(cls_all, modes).to_csv(
        out_dir / f"classification_comparison_{domain_key}.csv", index=False
    )

ckpt_df.to_csv(out_dir / "training_checkpoint_summary.csv", index=False)
selected.to_csv(out_dir / "selected_visualization_subjects.csv", index=False)
if ROI_PATCH_MODES and "selected_roi" in globals():
    selected_roi.to_csv(out_dir / "selected_visualization_subjects_roi.csv", index=False)

print(f"Exportado en {out_dir}")
print("
--- Resumen rápido ---")
for _domain_key, modes, domain_label in domain_buckets_for_modes(MODES, _REGISTRY):
    print(f"
[{domain_label}]")
    for mode in modes:
        r = recon_by_mode[mode]
        b = best_per_mode[best_per_mode["mode_label"] == MODE_LABELS[mode]].iloc[0]
        line = (
            f"  [{MODE_SHORT[mode]}] PSNR test={r['PSNR'].mean():.2f}±{r['PSNR'].std():.2f}, "
            f"mejor clf={b['model']} (bal.acc={b['balanced_accuracy']:.3f})"
        )
        if "PSNR_roi" in r.columns:
            line += f", PSNR_roi={r['PSNR_roi'].mean():.2f}"
        print(line)


## 5b. Visualización de latentes test (mejor clasificador)

Latentes **test** generados por inferencia DDIM (`sampled_latent_final`) del par **experimento + clasificador** con mayor `balanced_accuracy` en la sección 4.

Proyecciones **PCA**, **t-SNE** y **UMAP** en 2D y 3D (color = control / parkinson). Para t-SNE/UMAP se aplica PCA previa a 50 componentes cuando la dimensionalidad es alta.

Export: `result/stage2_roi3ch_wave_summary/latent_viz_best_classifier/`.


In [ ]:
from sklearn.metrics import davies_bouldin_score, silhouette_score

from code.embedding_analysis import label_name, projection_coordinates

try:
    import plotly.express as px

    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

best_cls_row = cls_all.sort_values(["roc_auc", "f1"], ascending=False).iloc[0]
best_mode = best_cls_row["mode"]
best_model = best_cls_row["model"]
best_cfg = COMPARISON[best_mode]

print(
    f"Mejor clasificador en test: [{MODE_SHORT[best_mode]}] {best_model} | "
    f"bal.acc={best_cls_row['balanced_accuracy']:.3f}, "
    f"precision={best_cls_row.get('precision', float('nan')):.3f}, "
    f"recall={best_cls_row.get('recall', float('nan')):.3f}, "
    f"f1={best_cls_row['f1']:.3f}"
)

X_lat, y_lat, ids_lat = load_test_latent_matrix_for(best_cfg["exp"], best_cfg["run_id"])
class_names = [label_name(y) for y in y_lat]
print(f"Latentes test: {X_lat.shape[0]} sujetos, dim={X_lat.shape[1]}")

PROJ_METHODS = ("pca", "tsne", "umap")
latent_proj: dict[tuple[str, int], pd.DataFrame] = {}
for method in PROJ_METHODS:
    for ndim in (2, 3):
        coords = projection_coordinates(X_lat, method, ndim)
        cols = [f"{method}_{i + 1}" for i in range(ndim)]
        df = pd.DataFrame(coords, columns=cols)
        df["ID"] = ids_lat
        df["label"] = y_lat
        df["class_name"] = class_names
        latent_proj[(method, ndim)] = df

viz_dir = REPORT_EXPORT_DIR / "latent_viz_best_classifier"
viz_dir.mkdir(parents=True, exist_ok=True)
for (method, ndim), df in latent_proj.items():
    df.to_csv(viz_dir / f"{method}_{ndim}d.csv", index=False)

summary = {
    "mode": best_mode,
    "mode_short": MODE_SHORT[best_mode],
    "mode_label": MODE_LABELS[best_mode],
    "model": best_model,
    "balanced_accuracy": float(best_cls_row["balanced_accuracy"]),
    "precision": float(best_cls_row.get("precision", np.nan)),
    "recall": float(best_cls_row.get("recall", np.nan)),
    "f1": float(best_cls_row["f1"]),
    "n_test_subjects": int(len(ids_lat)),
}
pd.Series(summary).to_json(viz_dir / "best_classifier_summary.json", indent=2)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, method in zip(axes, PROJ_METHODS):
    df2 = latent_proj[(method, 2)]
    sns.scatterplot(
        data=df2,
        x=f"{method}_1",
        y=f"{method}_2",
        hue="class_name",
        ax=ax,
        s=65,
        alpha=0.85,
    )
    ax.set_title(method.upper())
    ax.set_xlabel(f"{method}_1")
    ax.set_ylabel(f"{method}_2")
fig.suptitle(
    f"Latentes test DDIM — {MODE_SHORT[best_mode]} / {best_model} "
    f"(bal.acc={best_cls_row['balanced_accuracy']:.3f})",
    y=1.03,
)
plt.tight_layout()
plt.savefig(viz_dir / "projections_2d_panel.png", dpi=200, bbox_inches="tight")
plt.show()

for method in PROJ_METHODS:
    df3 = latent_proj[(method, 3)]
    if HAS_PLOTLY:
        fig3d = px.scatter_3d(
            df3,
            x=f"{method}_1",
            y=f"{method}_2",
            z=f"{method}_3",
            color="class_name",
            color_discrete_map={"control": "#3b6fd8", "parkinson": "#d83b5f"},
            hover_data=["ID", "label"],
            title=(
                f"{method.upper()} 3D — {MODE_SHORT[best_mode]} / {best_model} "
                f"(test, n={len(ids_lat)})"
            ),
            width=820,
            height=650,
        )
        fig3d.write_html(viz_dir / f"{method}_3d.html")
        fig3d.show()
    else:
        fig = plt.figure(figsize=(7, 6))
        ax = fig.add_subplot(111, projection="3d")
        for cn, part in df3.groupby("class_name"):
            ax.scatter(
                part[f"{method}_1"],
                part[f"{method}_2"],
                part[f"{method}_3"],
                label=cn,
                s=45,
                alpha=0.8,
            )
        ax.set_title(f"{method.upper()} 3D — {MODE_SHORT[best_mode]}")
        ax.legend()
        plt.tight_layout()
        plt.savefig(viz_dir / f"{method}_3d.png", dpi=200, bbox_inches="tight")
        plt.show()

sep_rows = []
for method in PROJ_METHODS:
    for ndim in (2, 3):
        cols = [f"{method}_{i + 1}" for i in range(ndim)]
        coords = latent_proj[(method, ndim)][cols].to_numpy()
        if len(np.unique(y_lat)) < 2 or len(y_lat) < 3:
            sil, db = np.nan, np.nan
        else:
            sil = float(silhouette_score(coords, y_lat))
            db = float(davies_bouldin_score(coords, y_lat))
        sep_rows.append({"method": method, "dims": ndim, "silhouette": sil, "davies_bouldin": db})
sep_df = pd.DataFrame(sep_rows).round(4)
sep_df.to_csv(viz_dir / "projection_separation_metrics.csv", index=False)
display(sep_df)
print(f"Export: {viz_dir}")


## 5c. Probabilidad Parkinson vs estadio NHY (mejor clasificador)

Para el mismo par **experimento / clasificador** seleccionado en §5b, se grafican las **probabilidades de Parkinson** (`score_*` en test) frente al estadio clínico:

| Eje X | Fuente |
|-------|--------|
| `control` | `COHORT=2` en `PPMI_MASTER_v2.xlsx` |
| `NHY1` / `NHY2` | columna `NHY` (PPMI) en sujetos PD (`COHORT=1`) |

Estadios derivados de `PATNO` + `NHY` en [`PPMI_MASTER_v2.xlsx`](../PPMI_MASTER_v2.xlsx). Export en `latent_viz_best_classifier/`.


In [ ]:
if "best_cls_row" not in globals():
    best_cls_row = cls_all.sort_values(["roc_auc", "balanced_accuracy"], ascending=False).iloc[0]
    best_mode = best_cls_row["mode"]
    best_model = best_cls_row["model"]
    best_cfg = COMPARISON[best_mode]

nhy_prob_df = build_parkinson_probability_by_nhy_stage(
    best_cfg["exp"],
    best_cfg["run_id"],
    best_model,
)
print(
    f"Clasificador: [{MODE_SHORT[best_mode]}] {best_model} | "
    f"n={len(nhy_prob_df)} test"
)
display(
    nhy_prob_df.groupby("stage", observed=True)["parkinson_probability"]
    .agg(["count", "mean", "std", "median", "min", "max"])
    .round(3)
)

viz_dir = REPORT_EXPORT_DIR / "latent_viz_best_classifier"
viz_dir.mkdir(parents=True, exist_ok=True)
nhy_prob_df.to_csv(viz_dir / "nhy_stage_parkinson_probabilities.csv", index=False)

stage_counts = nhy_prob_df["stage"].value_counts().reindex(NHY_STAGE_ORDER).fillna(0).astype(int)
x_labels = [f"{s}\n(n={stage_counts[s]})" for s in NHY_STAGE_ORDER]

fig, ax = plt.subplots(figsize=(7.5, 5))
sns.boxplot(
    data=nhy_prob_df,
    x="stage",
    y="parkinson_probability",
    order=list(NHY_STAGE_ORDER),
    palette={"control": "#3b6fd8", "NHY1": "#f0a202", "NHY2": "#d83b5f"},
    ax=ax,
)
sns.stripplot(
    data=nhy_prob_df,
    x="stage",
    y="parkinson_probability",
    order=list(NHY_STAGE_ORDER),
    color="0.25",
    size=5,
    alpha=0.65,
    jitter=0.18,
    ax=ax,
)
ax.set_xticks(range(len(NHY_STAGE_ORDER)))
ax.set_xticklabels(x_labels)
ax.set_ylim(-0.02, 1.02)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, alpha=0.7, label="umbral 0.5")
ax.set_xlabel("Estadio (PPMI NHY)")
ax.set_ylabel("Probabilidad Parkinson (test)")
ax.set_title(
    f"{MODE_SHORT[best_mode]} / {best_model} — "
    f"roc_auc={best_cls_row['roc_auc']:.3f}"
)
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(viz_dir / "nhy_stage_parkinson_probability_boxplot.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Export: {viz_dir}")


## 6. Diagnóstico sampling: `t` fijo vs DDIM

Compara los experimentos del reporte (`DIAG_MODES` por defecto = `MODES`) con **calibración latente coherente por experimento** (manifest → registry).

| Familia | Descripción |
|---------|-------------|
| **A. t fijo** | `z_gt` crudo → (opcional) `(z−μ)/σ` → ruido → un paso `predict_x0` → desnormalizar → métricas vs `z_gt` crudo |
| **B. DDIM** | Ruido en espacio de difusión del experimento → `sample()` → latente crudo |

`LATENT_CALIBRATION=1` solo si el experimento lo usó en entrenamiento (`run_manifest.json` o `experiments/registry.json`). Export en `REPORT_EXPORT_DIR` (`sampling_diagnosis_*`).

> **Importante:** re-carga el notebook desde disco si ves `UNet()` en el traceback. La carga del UNet usa `load_sampling_diag_models()` en esta sección.

In [ ]:
import importlib
from contextlib import contextmanager
from tqdm.auto import tqdm

import torch
from torch.utils.data import DataLoader

import json


def _registry_env_for_mode(mode_key: str) -> dict:
    reg = load_registry()
    entry = reg["experiments"].get(mode_key, {})
    env: dict = {}
    base_key = entry.get("base")
    if base_key and base_key in reg:
        env.update(reg[base_key])
    env.update(entry.get("overrides", {}))
    return env


def latent_calibration_for_mode(mode_key: str, exp: str) -> bool:
    """True iff this experiment trained with ILCLDM_LATENT_CALIBRATION=1."""
    manifest_path = repo_root() / "result" / exp.strip("/") / "run_manifest.json"
    if manifest_path.is_file():
        with open(manifest_path, encoding="utf-8") as f:
            manifest = json.load(f)
        cfg = manifest.get("config", {})
        if "LATENT_CALIBRATION" in cfg:
            return bool(cfg["LATENT_CALIBRATION"])
        env = manifest.get("env_overrides", {})
        if "ILCLDM_LATENT_CALIBRATION" in env:
            return str(env["ILCLDM_LATENT_CALIBRATION"]) == "1"
    env = _registry_env_for_mode(mode_key)
    if "ILCLDM_LATENT_CALIBRATION" in env:
        return str(env["ILCLDM_LATENT_CALIBRATION"]) == "1"
    return False


@contextmanager
def _latent_calibration_context(enabled: bool):
    import config

    prev = config.LATENT_CALIBRATION
    config.LATENT_CALIBRATION = bool(enabled)
    try:
        yield
    finally:
        config.LATENT_CALIBRATION = prev


@contextmanager
def _data_profile_for_mode(mode_key, comparison):
    import config
    from code.volume_shapes import clear_shape_cache
    from inference import apply_manifest_to_config

    saved = {
        "SKIP_CROP": config.SKIP_CROP,
        "ROI_VOLUME_SHAPE": getattr(config, "ROI_VOLUME_SHAPE", None),
        "whole_MRI": config.whole_MRI,
        "whole_Abeta": config.whole_Abeta,
        "latent_Abeta": config.latent_Abeta,
        "UNET_ROI_CHANNEL": config.UNET_ROI_CHANNEL,
        "LATENT_CALIBRATION": config.LATENT_CALIBRATION,
    }
    exp = comparison[mode_key]["exp"]
    if _mode_split_profile(mode_key) == "data_rois":
        apply_manifest_to_config(exp)
        clear_shape_cache()
    try:
        yield
    finally:
        for key, value in saved.items():
            setattr(config, key, value)
        clear_shape_cache()


def load_sampling_diag_models(mode_key, comparison, *, device, diag_mode="frozen"):
    import config
    from main import Diffusion
    from model import AAE, build_unet
    from utils import load_checkpoint
    from inference import (
        load_mri_encoder,
        resolve_mri_encoder_checkpoint,
        resolve_unet_build_config,
    )

    cfg = comparison[mode_key]
    exp = cfg["exp"]
    run_id = cfg["run_id"]
    root = repo_root()
    if _mode_split_profile(mode_key) == "data_rois":
        from inference import apply_manifest_to_config
        from code.volume_shapes import clear_shape_cache

        apply_manifest_to_config(exp)
        clear_shape_cache()
    checkpoint_aae = root / "result" / exp / "AAE.pth.tar"
    checkpoint_unet = root / "result" / exp / "Unet.pth.tar"
    mri_ckpt = None
    run_cfg_path = root / "result" / exp / "inference" / run_id / "run_config.json"
    if run_cfg_path.is_file():
        with open(run_cfg_path, encoding="utf-8") as f:
            run_cfg = json.load(f)
        checkpoint_aae = root / run_cfg["checkpoints"]["AAE"]
        checkpoint_unet = root / run_cfg["checkpoints"]["Unet"]
        mri_ckpt = root / run_cfg["checkpoints"]["MRI_encoder"]
    else:
        mri_ckpt = resolve_mri_encoder_checkpoint(exp, diag_mode, None)

    latent_calibration = latent_calibration_for_mode(mode_key, exp)
    use_roi_channel, in_channels = resolve_unet_build_config(exp, checkpoint_unet)
    aae = AAE().to(device).eval()
    load_checkpoint(str(checkpoint_aae), aae, None, config.learning_rate)
    unet = build_unet(in_channel=in_channels, use_roi_channel=use_roi_channel).to(device).eval()
    load_checkpoint(str(checkpoint_unet), unet, None, config.learning_rate)
    mri_encoder, _ = load_mri_encoder(diag_mode, mri_ckpt, device)
    with _latent_calibration_context(latent_calibration):
        diffusion = Diffusion()
        latent_mean = float(diffusion.latent_mean)
        latent_std = float(diffusion.latent_std)
    meta = {
        "exp": exp,
        "run_id": run_id,
        "checkpoint_unet": str(checkpoint_unet),
        "in_channels": in_channels,
        "use_roi_channel": use_roi_channel,
        "latent_calibration": latent_calibration,
        "latent_mean": latent_mean,
        "latent_std": latent_std,
    }
    return aae, unet, mri_encoder, diffusion, meta


import config
from dataset import TwoDataset
from main import Diffusion, batch_metric_scores
from utils import seed_torch
from inference import subject_id_from_name

DIAG_MODES = list(DIAG_MODES_DEFAULT)  # o ["h05_roi3ch"] para prueba rápida
DIAG_MODE = "frozen"
DIAG_SEED = 42
MAX_SUBJECTS = 10
FIXED_T_VALUES = [50, 200, 500, 700, 999]
DDIM_STEPS_LIST = [100]
USE_LABEL = True

device = torch.device(config.device)
seed_torch(DIAG_SEED)

print(
    f"Variantes: {[MODE_SHORT[m] for m in DIAG_MODES]} | "
    f"DDIM steps: {DDIM_STEPS_LIST}"
)


def _latent_metrics(z_hat: torch.Tensor, z_gt: torch.Tensor) -> dict:
    a = z_hat.detach().float().cpu().numpy().ravel()
    b = z_gt.detach().float().cpu().numpy().ravel()
    mse = float(np.mean((a - b) ** 2))
    mae = float(np.mean(np.abs(a - b)))
    corr = float(np.corrcoef(a, b)[0, 1]) if a.size > 1 else float("nan")
    return {"latent_mse": mse, "latent_mae": mae, "latent_corr": corr}


def _image_metrics(aae_model, spect_gt: torch.Tensor, z: torch.Tensor) -> dict:
    with torch.no_grad():
        pred = aae_model.decoder(z)
    psnr_v, ssim_v = batch_metric_scores(spect_gt.detach().cpu(), pred.detach().cpu())
    return {"PSNR": float(psnr_v), "SSIM": float(ssim_v)}


def _mri_condition(mri_enc, MRI: torch.Tensor) -> tuple[torch.Tensor, bool]:
    if mri_enc is not None:
        with torch.no_grad():
            return mri_enc(MRI), True
    return MRI, False


def _predict_noise(unet_model, x_t, cond, t, label, precomputed: bool):
    if USE_LABEL:
        return unet_model(x_t, cond, t, label, precomputed_condition=precomputed)
    return unet_model(x_t, cond, t, label=None, precomputed_condition=precomputed)


def _ddim_sample(unet_model, diff_model, cond, precomputed, label_batch, steps: int, seed: int) -> torch.Tensor:
    seed_torch(seed)

    def _model(x, y, t):
        return _predict_noise(unet_model, x, y, t, label_batch, precomputed)

    return diff_model.sample(_model, cond, sampling_steps=steps, eta=config.ddim_eta)


diag_rows = []
for mode_key in DIAG_MODES:
    with _data_profile_for_mode(mode_key, COMPARISON):
        data_root = data_root_for_mode(mode_key)
        test_ds = TwoDataset(
            root_MRI=str(data_root / "whole_MRI"),
            root_Abeta=str(data_root / "whole_Abeta"),
            task=str(repo_root() / "data_info" / "test.txt"),
            stage="test",
        )
        if MAX_SUBJECTS is not None:
            test_ds.images = test_ds.images[:MAX_SUBJECTS]
            test_ds.length_dataset = len(test_ds.images)
            test_ds.len = len(test_ds.images)

        loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

        aae, unet, mri_encoder, diffusion, diag_meta = load_sampling_diag_models(
            mode_key,
            COMPARISON,
            device=device,
            diag_mode=DIAG_MODE,
        )
        calib_tag = "calib" if diag_meta["latent_calibration"] else "raw"
        print(
            f"[{MODE_SHORT[mode_key]}] {MODE_LABELS[mode_key]} "
            f"in_ch={diag_meta['in_channels']} roi={diag_meta['use_roi_channel']} "
            f"latent={calib_tag} (mu={diag_meta['latent_mean']:.3f}, std={diag_meta['latent_std']:.3f}) "
            f"UNet={diag_meta['checkpoint_unet']}"
        )
        variant_short = MODE_SHORT[mode_key]
        variant_label = MODE_LABELS[mode_key]

        for batch_idx, (MRI, Abeta, name, label) in enumerate(
            tqdm(loader, desc=f"sampling diagnosis [{variant_short}]")
        ):
            sid = subject_id_from_name(name)
            MRI_t = torch.as_tensor(np.expand_dims(MRI, 1), dtype=torch.float32, device=device)
            spect_gt = torch.as_tensor(np.expand_dims(Abeta, 1), dtype=torch.float32, device=device)
            label_batch = label.to(device)

            with torch.no_grad():
                z_gt = aae.encoder(spect_gt)
                z_diff = diffusion.normalize_latent(z_gt)
                cond, precomputed = _mri_condition(mri_encoder, MRI_t)

                m_oracle = {**_latent_metrics(z_gt, z_gt), **_image_metrics(aae, spect_gt, z_gt)}
                diag_rows.append({
                    "ID": sid,
                    "variant": mode_key,
                    "variant_short": variant_short,
                    "variant_label": variant_label,
                    "method": "oracle_z_gt",
                    "t_or_steps": None,
                    "latent_calibration": diag_meta["latent_calibration"],
                    **m_oracle,
                })

                for t_val in FIXED_T_VALUES:
                    t = torch.full((1,), int(t_val), dtype=torch.long, device=device)
                    x_t, _noise = diffusion.noise_images(z_diff, t)
                    eps_pred = _predict_noise(unet, x_t, cond, t, label_batch, precomputed)
                    z_hat_diff = diffusion.predict_x0(x_t, t, eps_pred)
                    z_hat = diffusion.denormalize_latent(z_hat_diff)
                    m = {**_latent_metrics(z_hat, z_gt), **_image_metrics(aae, spect_gt, z_hat)}
                    diag_rows.append({
                        "ID": sid,
                        "variant": mode_key,
                        "variant_short": variant_short,
                        "variant_label": variant_label,
                        "method": "fixed_t",
                        "t_or_steps": t_val,
                        "latent_calibration": diag_meta["latent_calibration"],
                        **m,
                    })

                for steps in DDIM_STEPS_LIST:
                    z_gen = _ddim_sample(
                        unet, diffusion, cond, precomputed, label_batch, steps,
                        seed=DIAG_SEED + batch_idx,
                    )
                    m = {**_latent_metrics(z_gen, z_gt), **_image_metrics(aae, spect_gt, z_gen)}
                    diag_rows.append({
                        "ID": sid,
                        "variant": mode_key,
                        "variant_short": variant_short,
                        "variant_label": variant_label,
                        "method": "ddim",
                        "t_or_steps": steps,
                        "latent_calibration": diag_meta["latent_calibration"],
                        **m,
                    })

diag_df = pd.DataFrame(diag_rows)
diag_df["use_label"] = USE_LABEL
diag_df["experiment"] = diag_df["variant"].map(lambda k: COMPARISON[k]["exp"])
print(
    f"Filas: {len(diag_df)} | sujetos: {diag_df['ID'].nunique()} | "
    f"variantes: {[MODE_SHORT[m] for m in diag_df['variant'].unique()]}"
)
diag_df.head(8)


In [ ]:
def _method_label(row):
    if row["method"] == "oracle_z_gt":
        return "oracle z_gt"
    if row["method"] == "fixed_t":
        return f"t={int(row['t_or_steps'])}"
    return f"DDIM-{int(row['t_or_steps'])}"


diag_df["method_label"] = diag_df.apply(_method_label, axis=1)

summary = (
    diag_df.groupby(
        ["variant", "variant_short", "variant_label", "method", "t_or_steps", "method_label"],
        dropna=False,
    )
    .agg(
        n=("ID", "count"),
        latent_mse_mean=("latent_mse", "mean"),
        latent_mse_std=("latent_mse", "std"),
        latent_corr_mean=("latent_corr", "mean"),
        PSNR_mean=("PSNR", "mean"),
        PSNR_std=("PSNR", "std"),
        SSIM_mean=("SSIM", "mean"),
        SSIM_std=("SSIM", "std"),
    )
    .reset_index()
    .sort_values(["variant_short", "method", "t_or_steps"])
    .round(4)
)
summary


In [ ]:
plot_order = (
    ["oracle z_gt"]
    + [f"t={t}" for t in FIXED_T_VALUES]
    + [f"DDIM-{s}" for s in DDIM_STEPS_LIST]
)

for mode_key in DIAG_MODES:
    variant_short = MODE_SHORT[mode_key]
    plot_df = summary[summary["variant_short"] == variant_short].copy()
    plot_df = plot_df[plot_df["method_label"].isin(plot_order)].copy()
    plot_df["method_label"] = pd.Categorical(plot_df["method_label"], categories=plot_order, ordered=True)
    plot_df = plot_df.sort_values("method_label")

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    x = np.arange(len(plot_df))
    labels = plot_df["method_label"].astype(str)

    axes[0].bar(x, plot_df["latent_mse_mean"], yerr=plot_df["latent_mse_std"], capsize=3, color="steelblue")
    axes[0].set_title("MSE latente vs z_gt")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=45, ha="right")

    axes[1].bar(x, plot_df["PSNR_mean"], yerr=plot_df["PSNR_std"], capsize=3, color="seagreen")
    axes[1].set_title("PSNR imagen (decode → SPECT)")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=45, ha="right")

    axes[2].bar(x, plot_df["SSIM_mean"], yerr=plot_df["SSIM_std"], capsize=3, color="coral")
    axes[2].set_title("SSIM imagen")
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(labels, rotation=45, ha="right")

    fig.suptitle(
        f"Diagnóstico sampling — {variant_short} ({MODE_LABELS[mode_key]}) "
        f"(label={'sí' if USE_LABEL else 'no'})",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

    ft = plot_df[plot_df["method"] == "fixed_t"].sort_values("t_or_steps")
    if len(ft):
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(ft["t_or_steps"], ft["latent_mse_mean"], "o-", label="MSE latente")
        ax2 = ax.twinx()
        ax2.plot(ft["t_or_steps"], ft["PSNR_mean"], "s--", color="green", label="PSNR imagen")
        ax.set(xlabel="t", ylabel="MSE latente", title=f"1-paso desde z_gt ruidoso — {variant_short}")
        ax2.set_ylabel("PSNR imagen", color="green")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


In [ ]:
diag_out = REPORT_EXPORT_DIR
diag_out.mkdir(parents=True, exist_ok=True)
tag = f"ablations_label{int(USE_LABEL)}"
diag_df.to_csv(diag_out / f"sampling_diagnosis_{tag}_per_subject.csv", index=False)
summary.to_csv(diag_out / f"sampling_diagnosis_{tag}_summary.csv", index=False)
print(f"Exportado en {diag_out}")

for mode_key in DIAG_MODES:
    variant_short = MODE_SHORT[mode_key]
    sub = summary[summary["variant_short"] == variant_short]
    ddim_rows = sub[sub["method"] == "ddim"].sort_values("t_or_steps")
    ft_high = sub[(sub["method"] == "fixed_t") & (sub["t_or_steps"] >= 500)]
    print(f"\n--- [{variant_short}] {MODE_LABELS[mode_key]} ---")
    if len(ddim_rows):
        print(f"DDIM-100 MSE latente mean: {ddim_rows['latent_mse_mean'].iloc[0]:.4f}")
    if len(ft_high) and len(ddim_rows) and ft_high["latent_mse_mean"].mean() > ddim_rows["latent_mse_mean"].iloc[0] * 2:
        print("→ Denoising 1-paso falla en t altos: priorizar reentreno sobre más steps DDIM.")


**Valoración de superposición de mascara basal ganglia sobre un caso particular MRI y SPECT**

In [ ]:
# Sobreposición máscara basal ganglia sobre MRI y SPECT (sujeto 3102, cortes axiales z ∈ [63, 90])
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

if "DATA_ROOT" not in globals():
    here = Path.cwd().resolve()
    for candidate in (here, here.parent):
        if (candidate / "dataset.py").is_file() and (candidate / "viz_pairs.py").is_file():
            _root = candidate
            break
    else:
        raise RuntimeError(
            "No se encontró la raíz del repo. Ejecuta la celda de configuración o abre el notebook desde IL-CLDM/."
        )
    _root_str = str(_root)
    if _root_str not in sys.path:
        sys.path.insert(0, _root_str)
    if "code" in sys.modules and not hasattr(sys.modules["code"], "__path__"):
        del sys.modules["code"]
    DATA_ROOT = _root / "data"

from dataset import nifti_to_numpy
from viz_pairs import display_range

SUBJECT_ID = "3102"
Z_MIN, Z_MAX = 63, 90
MASK_PATH = DATA_ROOT / "mask_basal_ganglia.nii.gz"
MRI_PATH = DATA_ROOT / "whole_MRI" / f"{SUBJECT_ID}.nii.gz"
SPECT_PATH = DATA_ROOT / "whole_Abeta" / f"{SUBJECT_ID}.nii.gz"

bg_mask = nib.load(str(MASK_PATH)).get_fdata(dtype=np.float32)
bg_mask = (bg_mask > 0.5).astype(np.float32)

mri = nifti_to_numpy(str(MRI_PATH), zscore=True)
spect = nifti_to_numpy(str(SPECT_PATH), zscore=True)

if not (mri.shape == bg_mask.shape == spect.shape):
    raise ValueError(
        f"Shapes distintas: MRI {mri.shape}, SPECT {spect.shape}, máscara {bg_mask.shape}"
    )

z_slices = np.arange(Z_MIN, Z_MAX + 1)
print(
    f"Volumen {mri.shape} | cortes axiales z={Z_MIN}…{Z_MAX} ({len(z_slices)} slices) | "
    f"vox máscara total={int(bg_mask.sum()):,}"
)


def _overlay_slice(
    ax,
    vol,
    mask_vol,
    z,
    *,
    cmap="gray",
    vmin=None,
    vmax=None,
    mask_rgba=(1.0, 0.2, 0.2, 0.45),
):
    sl = vol[:, :, z].T
    msl = mask_vol[:, :, z].T > 0.5
    ax.imshow(sl, cmap=cmap, origin="lower", vmin=vmin, vmax=vmax)
    overlay = np.zeros((*sl.shape, 4), dtype=np.float32)
    overlay[msl, :3] = mask_rgba[:3]
    overlay[msl, 3] = mask_rgba[3]
    ax.imshow(overlay, origin="lower")
    ax.set_title(f"z={z}", fontsize=8)
    ax.axis("off")


def plot_mask_overlay_grid(vol, modality_label, *, symmetric=True, spect_hot=False):
    if spect_hot:
        vmin, vmax = display_range(vol, symmetric=False, percentiles=(5.0, 99.5))
        cmap = "hot"
    else:
        vmin, vmax = display_range(vol, symmetric=symmetric)
        cmap = "gray"

    ncols = 7
    nrows = int(np.ceil(len(z_slices) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.2 * ncols, 2.2 * nrows))
    axes = np.atleast_2d(axes)

    for idx, z in enumerate(z_slices):
        r, c = divmod(idx, ncols)
        _overlay_slice(axes[r, c], vol, bg_mask, z, cmap=cmap, vmin=vmin, vmax=vmax)

    for idx in range(len(z_slices), nrows * ncols):
        r, c = divmod(idx, ncols)
        axes[r, c].axis("off")

    fig.suptitle(
        f"Sujeto {SUBJECT_ID} — {modality_label} + máscara basal ganglia "
        f"(cortes axiales z∈[{Z_MIN}, {Z_MAX}])",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()


plot_mask_overlay_grid(mri, "MRI", symmetric=True)
plot_mask_overlay_grid(spect, "SPECT", spect_hot=True)

## Apéndice A. Proceso de difusión — baseline `exp_nm_s2_frozen_h05_roi3ch`

Objetivo: entender el **forward process** actual y valorar **schedules alternativos** (solo forward, sin reentrenar) antes de decidir si conviene cambiar `β_t` / `T`.

| Parte | Contenido |
|-------|-----------|
| **A.1 Schedule** | Curvas `β_t`, `ᾱ_t` y fracción de ruido `√(1-ᾱ_t)` |
| **A.2 Forward** | 2 sujetos test (1 control + 1 parkinson): `z_gt → x_t` en **latente** y **SPECT decodificado** |
| **A.3 Denoising** | Mismo baseline UNet: un paso `predict_x0` en `t ∈ {50, 200, 500, 700, 999}` con el **schedule de entrenamiento** |

**Notas:**
- Los presets alternativos (linear, distinto `β_end`, distinto `T`) se comparan con el **mismo ε** por sujeto.
- El UNet solo es válido con el schedule cosine `T=1000` con el que fue entrenado.
- Edita `SCHEDULE_PRESETS` y `FORWARD_T_VALUES` para explorar otras configuraciones.

In [ ]:
import json
from pathlib import Path

import torch
from torch.utils.data import DataLoader

import config
from dataset import TwoDataset, nifti_to_numpy
from main import Diffusion, batch_metric_scores
from model import AAE, build_unet
from utils import load_checkpoint, seed_torch
from inference import (
    checkpoint_paths,
    load_mri_encoder,
    resolve_mri_encoder_checkpoint,
    subject_id_from_name,
    resolve_unet_build_config,
)

APPENDIX_MODE = "h05_roi3ch"
APPENDIX_DIAG_MODE = "frozen"
APPENDIX_SEED = 42
FORWARD_T_VALUES = [50, 200, 500, 700, 999]

# Presets: el primero replica el entrenamiento actual (cosine, T=1000).
SCHEDULE_PRESETS = {
    "train_cosine": {
        "label": "Entrenamiento (cosine, T=1000)",
        "schedule": "cosine",
        "beta_start": 0.0001,
        "beta_end": 0.0200,
        "noise_steps": 1000,
    },
    "linear_default": {
        "label": "Linear β∈[1e-4, 2e-2]",
        "schedule": "linear",
        "beta_start": 0.0001,
        "beta_end": 0.0200,
        "noise_steps": 1000,
    },
    "cosine_low_beta": {
        "label": "Cosine β_end=0.01",
        "schedule": "cosine",
        "beta_start": 0.0001,
        "beta_end": 0.0100,
        "noise_steps": 1000,
    },
    "cosine_high_beta": {
        "label": "Cosine β_end=0.03",
        "schedule": "cosine",
        "beta_start": 0.0001,
        "beta_end": 0.0300,
        "noise_steps": 1000,
    },
    "linear_T500": {
        "label": "Linear T=500",
        "schedule": "linear",
        "beta_start": 0.0001,
        "beta_end": 0.0200,
        "noise_steps": 500,
    },
}

device = torch.device(config.device)
seed_torch(APPENDIX_SEED)


def resolve_stage1_aae_ckpt(mode_key: str) -> Path:
    cfg = COMPARISON[mode_key]
    run_cfg_path = (
        repo_root() / "result" / cfg["exp"] / "inference" / cfg["run_id"] / "run_config.json"
    )
    if run_cfg_path.exists():
        with open(run_cfg_path) as f:
            return repo_root() / json.load(f)["checkpoints"]["AAE"]
    env = os.getenv("ILCLDM_AAE_CHECKPOINT")
    if env:
        return Path(env)
    return repo_root() / "result/exp_nm_s1_spect/AAE.pth.tar"


def make_diffusion(preset: dict) -> Diffusion:
    return Diffusion(
        noise_steps=preset["noise_steps"],
        beta_start=preset["beta_start"],
        beta_end=preset["beta_end"],
        schedule=preset["schedule"],
    )


def effective_t(t: int, noise_steps: int) -> int:
    return int(min(max(t, 0), noise_steps - 1))


def forward_noisy_latent(diff: Diffusion, z_gt: torch.Tensor, t: int, eps: torch.Tensor) -> torch.Tensor:
    t_eff = effective_t(t, diff.noise_steps)
    t_batch = torch.full((z_gt.shape[0],), t_eff, dtype=torch.long, device=z_gt.device)
    sqrt_ah = torch.sqrt(diff.alpha_hat[t_batch])[:, None, None, None, None]
    sqrt_om = torch.sqrt(1.0 - diff.alpha_hat[t_batch])[:, None, None, None, None]
    return sqrt_ah * z_gt + sqrt_om * eps


def volume_mid_slice(vol: np.ndarray, axis: int = 2) -> np.ndarray:
    idx = vol.shape[axis] // 2
    if axis == 0:
        return vol[idx, :, :]
    if axis == 1:
        return vol[:, idx, :]
    return vol[:, :, idx]


def tensor_mid_slice(t: torch.Tensor, axis: int = 2) -> np.ndarray:
    return volume_mid_slice(t.detach().float().cpu().numpy().squeeze(), axis=axis)


def pick_two_test_subjects() -> pd.DataFrame:
    info = pd.read_csv(repo_root() / "data_info" / "data_info.csv", dtype={"ID": str})
    info = info[info["ID"].isin(TEST_IDS)].copy()
    rows = []
    for label_val, label_name in LABEL_NAMES.items():
        part = info[info["label"] == label_val].sort_values("ID")
        if part.empty:
            raise RuntimeError(f"No hay sujetos test con label={label_val}")
        rows.append(
            {
                "ID": part.iloc[0]["ID"],
                "label": int(label_val),
                "label_name": label_name,
            }
        )
    return pd.DataFrame(rows).reset_index(drop=True)


fwd_subjects = pick_two_test_subjects()
print("Sujetos apéndice (1 por clase):")
fwd_subjects

In [ ]:
# A.1 — Curvas del schedule (β_t, ᾱ_t, √(1-ᾱ_t))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
t_ref = np.arange(config.noiseSteps)

for key, preset in SCHEDULE_PRESETS.items():
    diff = make_diffusion(preset)
    steps = diff.noise_steps
    t_axis = np.arange(steps)
    beta = diff.beta.detach().cpu().numpy()
    alpha_hat = diff.alpha_hat.detach().cpu().numpy()
    noise_frac = np.sqrt(1.0 - alpha_hat)
    label = preset["label"]

    axes[0].plot(t_axis, beta, label=label, linewidth=1.5)
    axes[1].plot(t_axis, alpha_hat, label=label, linewidth=1.5)
    axes[2].plot(t_axis, noise_frac, label=label, linewidth=1.5)

for ax, title, ylab in zip(
    axes,
    [r"$\beta_t$", r"$\bar{\alpha}_t$", r"$\sqrt{1-\bar{\alpha}_t}$"],
    ["beta", "alpha_bar", "noise_fraction"],
):
    ax.set_title(title)
    ax.set_xlabel("t")
    ax.set_ylabel(ylab)
    ax.grid(True, alpha=0.3)

for t_mark in FORWARD_T_VALUES:
    if t_mark < config.noiseSteps:
        for ax in axes:
            ax.axvline(t_mark, color="0.75", linestyle=":", linewidth=0.8)

axes[0].legend(fontsize=7, loc="upper left")
fig.suptitle("Comparación de schedules (marcadores = t de visualización)", y=1.02)
plt.tight_layout()
plt.show()

# Tabla: nivel de ruido efectivo en los t fijos
rows = []
for key, preset in SCHEDULE_PRESETS.items():
    diff = make_diffusion(preset)
    for t in FORWARD_T_VALUES:
        t_eff = effective_t(t, diff.noise_steps)
        ah = float(diff.alpha_hat[t_eff].item())
        rows.append(
            {
                "preset": key,
                "label": preset["label"],
                "t_requested": t,
                "t_effective": t_eff,
                "alpha_bar": ah,
                "sqrt_noise": float(np.sqrt(1.0 - ah)),
            }
        )
schedule_t_table = pd.DataFrame(rows)
schedule_t_table.round(4)

In [ ]:
from dataset import crop

# A.2 — Forward process: z_gt → x_t (latente + SPECT decodificado)
aae_fwd = AAE().to(device).eval()
load_checkpoint(str(resolve_stage1_aae_ckpt(APPENDIX_MODE)), aae_fwd, None, config.learning_rate)

t_cols = [0] + FORWARD_T_VALUES
preset_keys = list(SCHEDULE_PRESETS.keys())


def plot_forward_grids(subject_row: pd.Series) -> None:
    sid = str(subject_row["ID"])
    spect_path = DATA_ROOT / "whole_Abeta" / f"{sid}.nii.gz"
    spect_np = crop(nifti_to_numpy(str(spect_path)))
    spect_gt = torch.as_tensor(np.expand_dims(spect_np, (0, 1)), dtype=torch.float32, device=device)

    with torch.no_grad():
        z_gt = aae_fwd.encoder(spect_gt)

    seed_torch(APPENDIX_SEED + int(sid))
    eps = torch.randn_like(z_gt)

    with torch.no_grad():
        z_gt_dec = aae_fwd.decoder(z_gt).cpu().numpy().squeeze()
    dec_lim = display_range(z_gt_dec, symmetric=False, percentiles=(5.0, 99.5))
    lat_lim = display_range(tensor_mid_slice(z_gt), symmetric=True)

    n_presets = len(preset_keys)
    n_t = len(t_cols)
    fig_lat, axes_lat = plt.subplots(n_presets, n_t, figsize=(2.3 * n_t, 2.3 * n_presets))
    fig_dec, axes_dec = plt.subplots(n_presets, n_t, figsize=(2.3 * n_t, 2.3 * n_presets))
    axes_lat = np.atleast_2d(axes_lat)
    axes_dec = np.atleast_2d(axes_dec)

    for i, pkey in enumerate(preset_keys):
        preset = SCHEDULE_PRESETS[pkey]
        diff = make_diffusion(preset)
        for j, t in enumerate(t_cols):
            if t == 0:
                z_t = z_gt
                ah = 1.0
                t_eff = 0
            else:
                t_eff = effective_t(t, diff.noise_steps)
                z_t = forward_noisy_latent(diff, z_gt, t, eps)
                ah = float(diff.alpha_hat[t_eff].item())

            with torch.no_grad():
                dec = aae_fwd.decoder(z_t).cpu().numpy().squeeze()

            ax_l = axes_lat[i, j]
            ax_d = axes_dec[i, j]
            ax_l.imshow(tensor_mid_slice(z_t).T, cmap="gray", origin="lower", vmin=lat_lim[0], vmax=lat_lim[1])
            ax_d.imshow(volume_mid_slice(dec).T, cmap="hot", origin="lower", vmin=dec_lim[0], vmax=dec_lim[1])
            ax_l.axis("off")
            ax_d.axis("off")

            if i == 0:
                title = "t=0" if t == 0 else f"t={t}"
                if t != 0 and t_eff != t:
                    title += f"→{t_eff}"
                ax_l.set_title(f"{title}\nᾱ={ah:.3f}", fontsize=8)
            if j == 0:
                ax_l.set_ylabel(preset["label"], fontsize=8)

    title_common = (
        f"Sujeto {sid} ({subject_row['label_name']}) — forward con ε fijo | "
        f"baseline {COMPARISON[APPENDIX_MODE]['exp']}"
    )
    fig_lat.suptitle(title_common + " — latente", y=1.02, fontsize=10)
    fig_dec.suptitle(title_common + " — SPECT decodificado", y=1.02, fontsize=10)
    plt.figure(fig_lat.number)
    plt.tight_layout()
    plt.show()
    plt.figure(fig_dec.number)
    plt.tight_layout()
    plt.show()


for _, subj in fwd_subjects.iterrows():
    plot_forward_grids(subj)

In [ ]:
# A.3 — Denoising UNet (schedule de entrenamiento): x_t → z_hat → decode
cfg = COMPARISON[APPENDIX_MODE]
exp = cfg["exp"]
run_cfg_path = repo_root() / "result" / exp / "inference" / cfg["run_id"] / "run_config.json"
if run_cfg_path.exists():
    with open(run_cfg_path) as f:
        run_cfg = json.load(f)
    checkpoint_aae = repo_root() / run_cfg["checkpoints"]["AAE"]
    checkpoint_unet = repo_root() / run_cfg["checkpoints"]["Unet"]
    mri_ckpt = repo_root() / run_cfg["checkpoints"]["MRI_encoder"]
else:
    checkpoint_aae, checkpoint_unet = checkpoint_paths(exp)
    checkpoint_aae = Path(checkpoint_aae)
    checkpoint_unet = Path(checkpoint_unet)
    mri_ckpt = resolve_mri_encoder_checkpoint(exp, APPENDIX_DIAG_MODE, None)

use_roi_channel, in_channels = resolve_unet_build_config(exp, checkpoint_unet)
unet = build_unet(in_channel=in_channels, use_roi_channel=use_roi_channel).to(device).eval()
load_checkpoint(str(checkpoint_unet), unet, None, config.learning_rate)
mri_encoder, _ = load_mri_encoder(APPENDIX_DIAG_MODE, mri_ckpt, device)
diffusion_train = make_diffusion(SCHEDULE_PRESETS["train_cosine"])
print(f"UNet: {checkpoint_unet} in_ch={in_channels} roi={use_roi_channel}")


def load_subject_batch(sid: str):
    ds = TwoDataset(
        root_MRI=str(DATA_ROOT / "whole_MRI"),
        root_Abeta=str(DATA_ROOT / "whole_Abeta"),
        task=str(repo_root() / "data_info" / "test.txt"),
        stage="test",
    )
    ds.images = [sid]
    ds.length_dataset = 1
    ds.len = 1
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    MRI, Abeta, name, label = next(iter(loader))
    MRI_t = torch.as_tensor(np.expand_dims(MRI, 1), dtype=torch.float32, device=device)
    spect_gt = torch.as_tensor(np.expand_dims(Abeta, 1), dtype=torch.float32, device=device)
    return MRI_t, spect_gt, label.to(device)


denoise_rows = []

for _, subj in fwd_subjects.iterrows():
    sid = str(subj["ID"])
    MRI_t, spect_gt, label_batch = load_subject_batch(sid)

    with torch.no_grad():
        z_gt = aae_fwd.encoder(spect_gt)
        if mri_encoder is not None:
            cond = mri_encoder(MRI_t)
            precomputed = True
        else:
            cond, precomputed = MRI_t, False

        z_gt_dec = aae_fwd.decoder(z_gt).cpu().numpy().squeeze()
    dec_lim = display_range(z_gt_dec, symmetric=False, percentiles=(5.0, 99.5))
    lat_lim = display_range(tensor_mid_slice(z_gt), symmetric=True)

    n_t = len(FORWARD_T_VALUES)
    fig, axes = plt.subplots(4, n_t, figsize=(2.5 * n_t, 9))
    row_labels = ["x_t (latente)", "z_hat (latente)", "SPECT GT", "SPECT z_hat"]

    for j, t_val in enumerate(FORWARD_T_VALUES):
        t = torch.full((1,), int(t_val), dtype=torch.long, device=device)
        with torch.no_grad():
            x_t, _ = diffusion_train.noise_images(z_gt, t)
            eps_pred = unet(x_t, cond, t, label_batch, precomputed_condition=precomputed)
            z_hat = diffusion_train.predict_x0(x_t, t, eps_pred)
            spect_hat = aae_fwd.decoder(z_hat).cpu().numpy().squeeze()
            spect_gt_np = spect_gt.cpu().numpy().squeeze()

        mse_lat = float(torch.mean((z_hat - z_gt) ** 2).item())
        psnr_v, ssim_v = batch_metric_scores(spect_gt_np, spect_hat)
        denoise_rows.append(
            {
                "ID": sid,
                "label_name": subj["label_name"],
                "t": t_val,
                "latent_mse": mse_lat,
                "PSNR": float(psnr_v),
                "SSIM": float(ssim_v),
            }
        )

        panels = [
            (tensor_mid_slice(x_t), "gray", lat_lim),
            (tensor_mid_slice(z_hat), "gray", lat_lim),
            (volume_mid_slice(spect_gt_np), "hot", dec_lim),
            (volume_mid_slice(spect_hat), "hot", dec_lim),
        ]
        for r, (img, cmap, lim) in enumerate(panels):
            ax = axes[r, j]
            ax.imshow(img.T, cmap=cmap, origin="lower", vmin=lim[0], vmax=lim[1])
            ax.axis("off")
            if r == 0:
                ax.set_title(f"t={t_val}", fontsize=9)
            if j == 0:
                ax.set_ylabel(row_labels[r], fontsize=8)

    fig.suptitle(
        f"Denoising 1-paso — {sid} ({subj['label_name']}) | {MODE_LABELS[APPENDIX_MODE]}",
        y=1.01,
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

denoise_df = pd.DataFrame(denoise_rows)
denoise_df.round(4)

In [ ]:
# Resumen denoising vs t (2 sujetos)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
metrics = ["latent_mse", "PSNR", "SSIM"]
for ax, metric in zip(axes, metrics):
    for label_name, part in denoise_df.groupby("label_name"):
        part = part.sort_values("t")
        ax.plot(part["t"], part[metric], marker="o", label=label_name)
    ax.set_xlabel("t")
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.suptitle("Aprendizaje del UNet baseline a lo largo de t (1 paso predict_x0)", y=1.02)
plt.tight_layout()
plt.show()

print(
    "Interpretación rápida:\n"
    "- Forward (A.2): si un schedule alternativo destruye demasiado pronto o deja mucha señal en t altos, "
    "puede ser más o menos adecuado para reentrenar.\n"
    "- Denoising (A.3): el UNet solo refleja el schedule de entrenamiento; compara calidad de "
    "reversión one-step frente a t."
)